In [3]:
import os
import requests
import pandas as pd
from tqdm import tqdm
import re

os.environ["OLLAMA_HOST"] = "ant7:11400"   # or ant7:11434, wherever ollama serve runs
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "localhost:11400")
OLLAMA_URL = f"http://{OLLAMA_HOST}/api/generate"

def generate_with_ollama(model: str, system_prompt: str, user_prompt: str) -> str:
    payload = {
        "model": model,
        "system": system_prompt,
        "prompt": user_prompt,
        "stream": False,
    }
    resp = requests.post(OLLAMA_URL, json=payload)
    resp.raise_for_status()
    text = resp.json()["response"]

    # Remove any <thought>...</thought> block if present
    text = re.sub(r"<thought>.*?</thought>\s*", "", text, flags=re.S)

    # Strip surrounding quotes / whitespace
    return text.strip().strip('"')

In [5]:
hotel_path = "/home/sohy47ma/ReviewProject_new/fake_reviews_prediction/Datasets/Hotel_Human_VS_HumanFake.csv"
df_hotel = pd.read_csv(hotel_path)

models = ["qwen2.5:32b"]
unique_hotels = df_hotel["Category"].unique()

num_per_hotel_per_model = 1  # adjust as you like

system_prompt = (
    "You write realistic hotel reviews. "
    "The reviews should look like genuine user opinions for analysis of fake-review detectors. "
    "Do not mention that you are an AI or that this is synthetic."
)

cg_rows = []

for model in models:
    for hotel in tqdm(unique_hotels, desc=f"Generating with {model}"):
        for i in range(num_per_hotel_per_model):
            user_prompt = (
                f"Write a hotel review for '{hotel}'. "
                f"Include comments about location, cleanliness, staff, and overall experience. "
                f"Length: 2–3 sentences."
            )
            text = generate_with_ollama(model, system_prompt, user_prompt)

            cg_rows.append({
                "Binary_label": "fake",          # will be fake_machine in your analysis
                "Category": hotel,               # context
                "domain": "Hotel",
                "text": text,
                "is_synthetic": 1,
                "source": model,                 # optional: which model generated it
            })

df_cg_hotel = pd.DataFrame(cg_rows)

out_cg_path = "/home/sohy47ma/ReviewProject_new/fake_reviews_prediction/Datasets/Hotel_CG_Reviews_final.csv"
df_cg_hotel.to_csv(out_cg_path, index=False)
print("Saved CG hotel reviews to:", out_cg_path)
print("CG rows:", len(df_cg_hotel))

Generating with qwen2.5:32b: 100%|██████████| 20/20 [00:50<00:00,  2.54s/it]

Saved CG hotel reviews to: /home/sohy47ma/ReviewProject_new/fake_reviews_prediction/Datasets/Hotel_CG_Reviews_final.csv
CG rows: 20


In [8]:
os.environ["OLLAMA_HOST"] = "ant2:11434"   # or ant7:11434, wherever ollama serve runs
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "localhost:11434")
OLLAMA_URL = f"http://{OLLAMA_HOST}/api/generate"
print("Using Ollama host:", OLLAMA_HOST)
print("Ollama URL:", OLLAMA_URL)

Using Ollama host: ant2:11434
Ollama URL: http://ant2:11434/api/generate


In [5]:
!hostname

ants
